# Andrea Pezzolla & Matteo Maruca
Progetto 2 DataScience, SUPSI, 2026

In [ ]:
# Import
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
# Dataframe
df = pd.read_csv("100097.csv", sep=';')
#FIXME usa tutti i dati come media annuale così si ha una vera percezione

In [17]:
# TRADUZIONI COLONNE 

# ── Rinomina colonne in italiano ──────────────────────────────
df = df.rename(columns={
    "Timestamp"                         : "timestamp",
    "Messung-ID"                        : "id_postazione",
    "Richtung ID"                       : "id_direzione",
    "Geschwindigkeit"                   : "velocita_kmh",
    "Zeit"                              : "ora",
    "Datum"                             : "data",
    "Datum und Zeit"                    : "data_ora",
    "Messbeginn"                        : "inizio_misurazione",
    "Messende"                          : "fine_misurazione",
    "Zone"                              : "zona_limite",
    "Ort"                               : "localita",
    "Richtung"                          : "direzione",
    "Koordinaten"                       : "coordinate_json",
    "Übertretungsquote"                 : "tasso_infrazioni_pct",
    "Geschwindigkeit V50"               : "v50_kmh",
    "Geschwindigkeit V85"               : "v85_kmh",
    "Strasse"                           : "via",
    "Hausnummer"                        : "numero_civico",
    "Fahrzeuge"                         : "n_veicoli",
    "Fahrzeuglänge"                     : "lunghezza_veicolo_m",
    "Kennzahlen pro Mess-Standort"      : "link_statistiche",
    "geo_point_2d"                      : "geo_point",
})

# ── Conversioni di tipo ───────────────────────────────────────
df["timestamp"]           = pd.to_datetime(df["timestamp"], utc=True)
df["inizio_misurazione"]  = pd.to_datetime(df["inizio_misurazione"])
df["fine_misurazione"]    = pd.to_datetime(df["fine_misurazione"])

# Estrai colonne utili da timestamp
df["anno"]         = df["timestamp"].dt.year
df["mese"]         = df["timestamp"].dt.month
df["giorno_sett"]  = df["timestamp"].dt.day_name(locale="it_IT.UTF-8")  # es. "Lunedì"
df["ora_intera"]   = df["timestamp"].dt.hour

# Zona limite come categoria
#df["zona_limite"] = df["zona_limite"].astype(int).astype(str).apply(lambda x: f"Zona {x}")

print("Colonne rinominate:")
print(df.columns.tolist())
print(f"\nDimensioni: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(df.dtypes)

Colonne rinominate:
['timestamp', 'id_postazione', 'id_direzione', 'velocita_kmh', 'ora', 'data', 'data_ora', 'inizio_misurazione', 'fine_misurazione', 'zona_limite', 'localita', 'direzione', 'coordinate_json', 'tasso_infrazioni_pct', 'v50_kmh', 'v85_kmh', 'via', 'numero_civico', 'n_veicoli', 'lunghezza_veicolo_m', 'link_statistiche', 'geo_point', 'anno', 'mese', 'giorno_sett', 'ora_intera', 'cat_veicolo', 'lat', 'lon']

Dimensioni: 5,422,501 righe × 29 colonne
timestamp               datetime64[us, UTC]
id_postazione                         int64
id_direzione                          int64
velocita_kmh                        float64
ora                                     str
data                                    str
data_ora                                str
inizio_misurazione           datetime64[us]
fine_misurazione             datetime64[us]
zona_limite                             str
localita                                str
direzione                               str
coordi

In [ ]:
# ── ANALISI 1 ────────────────────────────────────────────────
# Distribuzione delle Velocità per Zona Limite
# ────────────────────────────────────────────────────────────

fig1 = px.histogram(
    df,
    x="velocita_kmh",
    color="zona_limite",
    nbins=60,
    barmode="overlay",
    opacity=0.75,
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Distribuzione delle Velocità per Zona Limite",
    labels={
        "velocita_kmh": "Velocità (km/h)",
        "count":        "Numero misurazioni",
        "zona_limite":  "Zona"
    }
)
fig1.show()

In [ ]:
# ── ANALISI 2 ────────────────────────────────────────────────
# Top 15 postazioni per tasso medio di infrazioni
# ────────────────────────────────────────────────────────────

top_infrazioni = (
    df.groupby(["localita", "via"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .sort_values("tasso_infrazioni_pct", ascending=False)
    .head(15)
)

fig2 = px.bar(
    top_infrazioni,
    x="tasso_infrazioni_pct",
    y="via",
    orientation="h",
    color="tasso_infrazioni_pct",
    color_continuous_scale="Oranges",
    hover_data=["localita"],
    title="Top 15 Postazioni per Tasso di Infrazioni (%)",
    labels={
        "tasso_infrazioni_pct": "Tasso infrazioni (%)",
        "via":                  "Via",
        "localita":             "Località"
    }
)
fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False
)
fig2.show()

In [ ]:
# ── ANALISI 3 ────────────────────────────────────────────────
# Mappa di calore – Velocità media per giorno e ora
# ────────────────────────────────────────────────────────────

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

pivot = (
    df.groupby(["giorno_sett", "ora_intera"])["velocita_kmh"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="velocita_kmh")
    .reindex(ordine_giorni)
)

fig3 = px.imshow(
    pivot,
    color_continuous_scale="inferno",
    aspect="auto",
    title="Velocità Media per Giorno della Settimana e Ora",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Velocità media (km/h)"
    }
)
fig3.update_layout(
    xaxis_title="Ora del giorno",
    yaxis_title=""
)
fig3.show() 

In [ ]:
# ── ANALISI 4 ────────────────────────────────────────────────
# Mappa di calore – infrazioni per giorno e ora (variante)
# Stessa struttura ma su tasso_infrazioni_pct
# ────────────────────────────────────────────────────────────

pivot_inf = (
    df.groupby(["giorno_sett", "ora_intera"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="tasso_infrazioni_pct")
    .reindex(ordine_giorni)
)

fig4 = px.imshow(
    pivot_inf,
    color_continuous_scale="inferno",
    aspect="auto",
    title="Tasso Infrazioni Medio per Giorno e Ora (%)",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Infrazioni (%)"
    }
)
fig4.update_layout(
    xaxis_title="Ora del giorno",
    yaxis_title=""
)
fig4.show()

In [ ]:
# ── ANALISI 5 ────────────────────────────────────────────────
# Scatter V50 vs V85 per postazione
# Slide 7 – Correlazione tra velocità mediana e 85° percentile
# ────────────────────────────────────────────────────────────

agg_postazioni = (
    df.groupby(["id_postazione", "via", "zona_limite"])
    .agg(
        v50=("v50_kmh",             "mean"),
        v85=("v85_kmh",             "mean"),
        infrazioni=("tasso_infrazioni_pct", "mean"),
        n_veicoli=("n_veicoli",     "mean")
    )
    .reset_index()
)

fig5 = px.scatter(
    agg_postazioni,
    x="v50",
    y="v85",
    color="zona_limite",
    size="infrazioni",
    hover_data=["via", "infrazioni", "n_veicoli"],
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Correlazione V50 – V85 per Postazione",
    labels={
        "v50":        "V50 – velocità mediana (km/h)",
        "v85":        "V85 – 85° percentile (km/h)",
        "zona_limite":"Zona",
        "infrazioni": "Tasso infrazioni (%)"
    }
)
fig5.show()

In [ ]:
# ── ANALISI 6 ────────────────────────────────────────────────
# Distribuzione velocità per tipo di veicolo (lunghezza)
# Boxplot – velocita_kmh per categoria lunghezza veicolo
# ────────────────────────────────────────────────────────────

# Categorizza la lunghezza del veicolo
bins   = [0, 3.5, 6, 10, 100]
labels = ["Auto (<3.5m)", "Furgone (3.5–6m)", "Camion (6–10m)", "Autoarticolato (>10m)"]

df["cat_veicolo"] = pd.cut(
    df["lunghezza_veicolo_m"],
    bins=bins,
    labels=labels,
    right=True
)

fig6 = px.box(
    df,
    x="cat_veicolo",
    y="velocita_kmh",
    color="zona_limite",
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Distribuzione Velocità per Categoria di Veicolo e Zona",
    labels={
        "cat_veicolo":  "Categoria veicolo",
        "velocita_kmh": "Velocità (km/h)",
        "zona_limite":  "Zona"
    }
)
fig6.show()

In [ ]:
# ── ANALISI 7 ────────────────────────────────────────────────
# Trend mensile – velocità media e infrazioni nel tempo
# Line chart – evoluzione mese per mese
# ────────────────────────────────────────────────────────────

trend = (
    df.groupby(["anno", "mese"])
    .agg(
        vel_media  = ("velocita_kmh",          "mean"),
        infrazioni = ("tasso_infrazioni_pct",  "mean")
    )
    .reset_index()
)
 
# FIX: costruisci la data come stringa "YYYY-MM-01" e poi converti
trend["periodo"] = pd.to_datetime(
    trend["anno"].astype(str) + "-" + trend["mese"].astype(str).str.zfill(2) + "-01"
)
 
fig7 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Velocità media (km/h)", "Tasso infrazioni medio (%)")
)
fig7.add_trace(
    go.Scatter(
        x=trend["periodo"], y=trend["vel_media"],
        mode="lines+markers", name="Velocità media",
        line=dict(color="#00B4D8", width=2)
    ),
    row=1, col=1
)
fig7.add_trace(
    go.Scatter(
        x=trend["periodo"], y=trend["infrazioni"],
        mode="lines+markers", name="Tasso infrazioni",
        line=dict(color="#E07B39", width=2)
    ),
    row=2, col=1
)
fig7.update_layout(
    title="Trend Mensile – Velocità Media e Tasso Infrazioni",
    plot_bgcolor="#CACBDE",
    showlegend=True
)
fig7.show()

In [ ]:
# ── ANALISI 8 ────────────────────────────────────────────────
# Mappa interattiva delle postazioni (Plotly mapbox)
# Slide 8 futura – punti colorati per tasso infrazioni
# ────────────────────────────────────────────────────────────
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)
 
mappa = (
    df.groupby(["id_postazione", "via", "zona_limite", "lat", "lon"])
    .agg(
        infrazioni = ("tasso_infrazioni_pct", "mean"),
        vel_media  = ("velocita_kmh",          "mean"),
        n_veicoli  = ("n_veicoli",             "mean")
    )
    .reset_index()
)

fig8 = px.scatter_map(
    mappa,
    lat="lat",
    lon="lon",
    color="infrazioni",
    size="n_veicoli",
    size_max=20,
    hover_name="via",
    hover_data={"infrazioni": ":.1f", "vel_media": ":.1f", "zona_limite": True},
    color_continuous_scale="inferno",
    zoom=12,
    map_style="carto-positron",
    title="Mappa Postazioni – Tasso di Infrazioni (%)",
    labels={
        "infrazioni": "Infrazioni (%)",
        "vel_media":  "Vel. media (km/h)",
        "n_veicoli":  "Veicoli"
    }
)
fig8.update_layout(
    margin=dict(l=0, r=0, t=40, b=0)
)
fig8.show()

# Grafici Maru

## Grafico scatter geo che mostra l'omogeneità delle registrazioni nel canton basilea

In [29]:
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)

#df['zona_limite'] = df['zona_limite'].str.extract(r'(\d+)')[0].astype(int)

df['Superamento'] = df['velocita_kmh'] - df['zona_limite']
idx_max = df.groupby(['lat', 'lon'])['Superamento'].idxmax()
df_peggiori = df.loc[idx_max].reset_index(drop=True)

fig = px.scatter_mapbox(
    df_peggiori,
    lat="lat",
    lon="lon",
    zoom=12,                        # Livello di zoom ideale per una città/cantone
    hover_name='zona_limite',
    center={"lat": 47.5596, "lon": 7.5886},  # Coordinate del centro di Basilea
    title="Misurazioni di Velocità - Canton Basilea",
    size="Superamento",
    color="velocita_kmh",
    color_continuous_scale="Reds"
)

fig.update_layout(mapbox_style="carto-positron")

fig.show()

/var/folders/41/zydj4q6x2hgf50rklr4zfwl00000gn/T/ipykernel_63751/2849294073.py:13: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


IL grafico mostra il superamento massimo registrato da ogni radar e si vede bene perche con il colore vediamo la velocita a cui andava e invece con la grandezza quanto ha superato il limite.